# Firecrawl Local (CPU · Rust v2 API + Native-Sandbox Chrome + Drive Binary Cache)

Serves `firecrawl-local` (Firecrawl v2 HTTP API: `/v2/scrape`, `/v2/map`, `/v2/crawl`, `/v2/batch/scrape`, `/health`) on a zero-GPU Colab CPU runtime (`gpu=None`, optional `--high-mem`) behind Caddy Basic Auth + Bearer token auth. Compiles the single Rust binary once on local SSD and caches it to Google Drive (`$CT_DATA_DIR/bin/`) for instant reuse on subsequent sessions, running the server as an unprivileged `firecrawl` user with native-sandbox headless Chrome for JavaScript rendering.

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "firecrawl-local")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "3002"))
AUTH         = os.environ.get("CT_AUTH", "caddy")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "18080"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = os.environ.get("CT_WORK_DIR", "/content/ct")             # logs + runtime configs (0700)
CACHE_DIR = os.environ.get("CT_CACHE_DIR", "")
BIN_DIR = os.environ.get("CT_BIN_DIR", "/usr/local/bin")
os.makedirs(WORK, mode=0o700, exist_ok=True)
os.chmod(WORK, 0o700)
os.makedirs(BIN_DIR, exist_ok=True)
if BIN_DIR not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = BIN_DIR + os.pathsep + os.environ.get("PATH", "")
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
MOUNT_DRIVE = os.environ.get("CT_MOUNT_DRIVE", "1") != "0"
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not MOUNT_DRIVE or not os.path.isdir("/content/drive/MyDrive"):   # Drive not mounted -> local fallback
    if DATA_DIR.startswith("/content/drive/"):
        DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to <WORK>/<name>.log (0600)."""
    lp = f"{WORK}/{name}.log"
    fd = os.open(lp, os.O_WRONLY | os.O_CREAT | os.O_APPEND, 0o600)
    os.chmod(lp, 0o600)
    with os.fdopen(fd, "a") as logf:   # child keeps its own dup of the fd
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

CADDY_VERSION = "2.10.2"

def ct_write_private_text(path, content):
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    os.chmod(path, 0o600)
    with os.fdopen(fd, "w", encoding="utf-8") as handle:
        handle.write(content)
        handle.flush()
        os.fsync(handle.fileno())

def ct_proc_start_time(pid):
    try:
        fields = open(f"/proc/{int(pid)}/stat", "r", encoding="utf-8", errors="replace").read().rsplit(")", 1)[1].split()
        if fields[0] == "Z":
            return None
        return fields[19]
    except (OSError, IndexError, ValueError):
        return None

def ct_stop_marked_proc(name, timeout=10):
    import signal
    marker = f"{WORK}/{name}.proc.json"
    if not os.path.exists(marker):
        return
    try:
        info = json.loads(open(marker, "r", encoding="utf-8").read())
        pid = int(info.get("pid") or 0)
        start_time = info.get("start_time")
    except (OSError, ValueError, TypeError):
        pid, start_time = 0, None
    if pid > 0 and start_time and ct_proc_start_time(pid) == start_time:
        for sig in (signal.SIGTERM, signal.SIGKILL):
            if ct_proc_start_time(pid) != start_time:
                break
            try:
                os.killpg(os.getpgid(pid), sig)
            except (ProcessLookupError, PermissionError, OSError):
                try:
                    os.kill(pid, sig)
                except (ProcessLookupError, OSError):
                    break
            deadline = time.monotonic() + timeout
            while time.monotonic() < deadline:
                if ct_proc_start_time(pid) != start_time:
                    break
                time.sleep(0.1)
    try:
        os.unlink(marker)
    except OSError:
        pass

def ct_spawn_marked(name, args, env=None, cwd=None):
    ct_stop_marked_proc(name)
    proc = ct_spawn(name, args, env=env, cwd=cwd)
    marker = f"{WORK}/{name}.proc.json"
    ct_write_private_text(marker, json.dumps({"pid": proc.pid, "start_time": ct_proc_start_time(proc.pid)}))
    return proc

def _ct_file_sha256(path):
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(65536)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def ct_ensure_caddy():
    import hashlib, platform, tarfile, tempfile
    arch = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")
    dest = f"{BIN_DIR}/caddy"
    status_label = "miss"
    manifest = f"{CACHE_DIR}/manifests/caddy.json" if CACHE_DIR else ""
    if os.path.exists(dest):
        if not manifest:
            return dest
        try:
            meta = json.loads(open(manifest, "r", encoding="utf-8").read())
            if meta.get("artifact") == "caddy" and meta.get("version") == CADDY_VERSION and meta.get("sha256") == _ct_file_sha256(dest):
                print("CT_CACHE_EVENT artifact=caddy status=hit", flush=True)
                return dest
        except (OSError, ValueError):
            pass
        try:
            os.unlink(dest)
        except OSError:
            pass
        status_label = "corrupt_rebuilt"
    asset = f"caddy_{CADDY_VERSION}_linux_{arch}.tar.gz"
    checks_asset = f"caddy_{CADDY_VERSION}_checksums.txt"
    base = f"https://github.com/caddyserver/caddy/releases/download/v{CADDY_VERSION}/"
    os.makedirs(BIN_DIR, exist_ok=True)
    with tempfile.TemporaryDirectory() as tmp:
        tgz = os.path.join(tmp, asset)
        with urllib.request.urlopen(base + asset, timeout=60) as src, open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        checks = urllib.request.urlopen(base + checks_asset, timeout=60).read().decode("utf-8")
        expected = next((line.split()[0] for line in checks.splitlines() if line.split() and line.split()[-1].lstrip("*") == asset), None)
        algo = {64: hashlib.sha256, 128: hashlib.sha512}.get(len(expected or ""))
        if algo is None or algo(open(tgz, "rb").read()).hexdigest() != expected:
            raise RuntimeError(f"Caddy checksum verification failed for {asset}")
        partial = f"{dest}.{os.getpid()}.partial"
        try:
            with tarfile.open(tgz) as tar:
                entry = tar.getmember("caddy")
                with tar.extractfile(entry) as src, open(partial, "wb") as out:
                    shutil.copyfileobj(src, out)
            os.chmod(partial, 0o755)
            os.replace(partial, dest)
        finally:
            if os.path.exists(partial):
                os.unlink(partial)
    if manifest:
        os.makedirs(os.path.dirname(manifest), exist_ok=True)
        tmp_m = f"{manifest}.{os.getpid()}.tmp"
        with open(tmp_m, "w", encoding="utf-8") as mf:
            mf.write(json.dumps({
                "artifact": "caddy",
                "version": CADDY_VERSION,
                "sha256": _ct_file_sha256(dest),
                "bin_path": dest,
                "verified_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            }, indent=2) + "\n")
        os.replace(tmp_m, manifest)
        print(f"CT_CACHE_EVENT artifact=caddy status={status_label}", flush=True)
    return dest

def ct_stop_proc(proc, timeout=15):
    """Terminate a ct_spawn process group, escalating to SIGKILL after a bounded wait."""
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_gpu_stat(query="memory.used,memory.total,utilization.gpu"):
    """nvidia-smi summary that never raises (CPU runtimes / transient driver errors)."""
    try:
        return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                                       text=True, stderr=subprocess.DEVNULL, timeout=20).strip()
    except (OSError, subprocess.SubprocessError):
        return "cpu/no-gpu"

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    """Wait for the public endpoint; auth failures are fatal, network errors are retried."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:   # DNS / edge propagation / TLS warm-up
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()            # RuntimeError propagates: never report CT_READY with broken auth
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=60, grace=900, dead_markers=(), heartbeat=None, heartbeat_interval=None):
    """Supervise PROCS forever: restart dead processes, debounce health failures after reloads."""
    started = time.monotonic()
    last_heartbeat = started
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        now_m = time.monotonic()
        if heartbeat is not None and (heartbeat_interval is None or (now_m - last_heartbeat) >= heartbeat_interval):
            last_heartbeat = now_m
            try:
                heartbeat(int(now_m - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        partial = f"{path}.{os.getpid()}.partial"
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with urllib.request.urlopen(url, timeout=60) as src, open(partial, "wb") as dst:
            shutil.copyfileobj(src, dst)
        os.chmod(partial, os.stat(partial).st_mode | stat.S_IEXEC)
        os.replace(partial, path)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
# @title 1. Setup -- restore/compile Firecrawl Local binary (Drive-cached) & stage Chrome sandbox
import sys
from pathlib import Path

FC_WORK = Path(os.environ.get('CT_FIRECRAWL_WORK_DIR', '/content/firecrawl-local'))
FC_WORK.mkdir(parents=True, exist_ok=True)
(FC_WORK / 'firecrawl_local_setup.py').write_text('"""Build, cache, and runtime staging helper for Firecrawl Local on Colab CPU VMs.\n\nStandard-library only so local unit tests can import and exercise all caching,\nintegrity-verification, and command-building logic without external packages.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport tempfile\nimport time\nimport urllib.error\nimport urllib.request\nfrom typing import Callable\n\nFIRECRAWL_REPO_URL = "https://github.com/firecrawl/firecrawl.git"\nGITHUB_COMMITS_API = "https://api.github.com/repos/firecrawl/firecrawl/commits"\nCHROME_DEB_URL = "https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb"\n\n_HEX_SHA_RE = re.compile(r"^[0-9a-fA-F]{7,40}$")\n_SAFE_TOKEN_RE = re.compile(r"[^A-Za-z0-9._-]+")\n_GLIBC_RE = re.compile(r"(\\d+\\.\\d+)")\n\n\ndef sanitize_token(value: str, fallback: str = "default") -> str:\n    cleaned = _SAFE_TOKEN_RE.sub("-", (value or "").strip()).strip("-.")\n    return cleaned[:64] if cleaned else fallback\n\n\ndef detect_arch() -> str:\n    return sanitize_token(platform.machine() or "x86_64", "x86_64")\n\n\ndef detect_glibc_version() -> str:\n    try:\n        out = subprocess.check_output(["ldd", "--version"], text=True, stderr=subprocess.STDOUT, timeout=10)\n        first = out.splitlines()[0] if out else ""\n        # Example: "ldd (Ubuntu GLIBC 2.39-0ubuntu8.9) 2.39"\n        matches = _GLIBC_RE.findall(first)\n        if matches:\n            return matches[-1]\n    except (OSError, subprocess.SubprocessError):\n        pass\n    return "unknown"\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as fh:\n        while True:\n            chunk = fh.read(1024 * 1024)\n            if not chunk:\n                break\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef verify_cached_binary(binary_path: Path, sha_path: Path) -> bool:\n    if not binary_path.is_file() or binary_path.stat().st_size < 1024:\n        return False\n    if not sha_path.is_file():\n        return False\n    try:\n        expected = sha_path.read_text(encoding="utf-8").strip().split()[0].lower()\n    except (OSError, IndexError):\n        return False\n    if len(expected) != 64:\n        return False\n    return sha256_file(binary_path).lower() == expected\n\n\ndef _atomic_copy(src: Path, dst: Path, mode: int | None = None) -> None:\n    dst.parent.mkdir(parents=True, exist_ok=True)\n    fd, tmp_name = tempfile.mkstemp(prefix=f".{dst.name}.", suffix=".tmp", dir=str(dst.parent))\n    os.close(fd)\n    tmp = Path(tmp_name)\n    try:\n        shutil.copyfile(src, tmp)\n        if mode is not None:\n            os.chmod(tmp, mode)\n        os.replace(tmp, dst)\n    except BaseException:\n        tmp.unlink(missing_ok=True)\n        raise\n\n\ndef _atomic_write_text(dst: Path, text: str) -> None:\n    dst.parent.mkdir(parents=True, exist_ok=True)\n    fd, tmp_name = tempfile.mkstemp(prefix=f".{dst.name}.", suffix=".tmp", dir=str(dst.parent))\n    tmp = Path(tmp_name)\n    try:\n        with os.fdopen(fd, "w", encoding="utf-8") as fh:\n            fh.write(text)\n        os.replace(tmp, dst)\n    except BaseException:\n        tmp.unlink(missing_ok=True)\n        raise\n\n\ndef _fetch_latest_path_commit(ref: str, subpath: str, timeout: int = 10) -> tuple[str, str] | None:\n    url = f"{GITHUB_COMMITS_API}?path={subpath}&sha={urllib.request.quote(ref, safe=\'\')}&per_page=1"\n    req = urllib.request.Request(\n        url,\n        headers={\n            "Accept": "application/vnd.github+json",\n            "User-Agent": "colabllm-firecrawl-local/1.0",\n        },\n    )\n    try:\n        with urllib.request.urlopen(req, timeout=timeout) as resp:\n            items = json.loads(resp.read().decode("utf-8"))\n        if isinstance(items, list) and items:\n            sha = str(items[0].get("sha") or "").strip()\n            date = str(((items[0].get("commit") or {}).get("committer") or {}).get("date") or "")\n            if _HEX_SHA_RE.match(sha):\n                return sha, date\n    except (OSError, urllib.error.URLError, ValueError, KeyError):\n        return None\n    return None\n\n\ndef resolve_source_spec(ref: str = "main", *, log: Callable[[str], None] = print) -> dict[str, str]:\n    """Resolve `ref` into a deterministic build & cache spec.\n\n    Supported formats:\n      - `crates:<version>` (e.g. `crates:0.1.0` or `crates:latest`)\n      - Git branch / tag / commit SHA (default: `main`)\n\n    For git branches like `main`, we query the latest commit touching\n    `apps/local` or `apps/api/native/src` so unrelated monorepo commits\n    never invalidate the Google Drive binary cache.\n    """\n    raw = (ref or "main").strip()\n    if raw.startswith("crates:"):\n        ver = raw.split(":", 1)[1].strip() or "latest"\n        return {\n            "kind": "crates",\n            "ref": ver,\n            "cache_tag": f"crates-{sanitize_token(ver, \'latest\')}",\n        }\n\n    if _HEX_SHA_RE.match(raw):\n        return {\n            "kind": "git",\n            "ref": raw,\n            "checkout_ref": raw,\n            "cache_tag": f"git-{raw[:12].lower()}",\n        }\n\n    # Branch or tag: check latest commit affecting apps/local or apps/api/native/src.\n    candidates: list[tuple[str, str]] = []\n    for subpath in ("apps/local", "apps/api/native/src"):\n        hit = _fetch_latest_path_commit(raw, subpath)\n        if hit:\n            candidates.append(hit)\n    if candidates:\n        # Pick the most recent committer timestamp across both paths.\n        candidates.sort(key=lambda item: item[1], reverse=True)\n        latest_sha = candidates[0][0]\n        return {\n            "kind": "git",\n            "ref": raw,\n            "checkout_ref": raw,\n            "cache_tag": f"git-{latest_sha[:12].lower()}",\n        }\n\n    # Fallback: git ls-remote (works even if GitHub REST API is rate-limited).\n    try:\n        out = subprocess.check_output(\n            ["git", "ls-remote", FIRECRAWL_REPO_URL, raw],\n            text=True,\n            stderr=subprocess.DEVNULL,\n            timeout=20,\n        ).strip()\n        if out:\n            sha = out.splitlines()[0].split()[0]\n            if _HEX_SHA_RE.match(sha):\n                return {\n                    "kind": "git",\n                    "ref": raw,\n                    "checkout_ref": raw,\n                    "cache_tag": f"git-{sha[:12].lower()}",\n                }\n    except (OSError, subprocess.SubprocessError) as err:\n        log(f"Warning: could not resolve git ref {raw!r} remotely: {err}")\n\n    return {\n        "kind": "git",\n        "ref": raw,\n        "checkout_ref": raw,\n        "cache_tag": f"git-{sanitize_token(raw, \'main\')}",\n    }\n\n\ndef make_cache_key(cache_tag: str, arch: str | None = None, glibc: str | None = None) -> str:\n    a = sanitize_token(arch or detect_arch(), "x86_64")\n    g = sanitize_token(glibc or detect_glibc_version(), "unknown")\n    return f"firecrawl-local-{sanitize_token(cache_tag, \'main\')}-{a}-glibc{g}"\n\n\ndef _find_latest_fallback_cache(cache_dir: Path, arch: str, glibc: str) -> tuple[Path, Path, str] | None:\n    pointer = cache_dir / f"latest-{arch}-glibc{glibc}.json"\n    if pointer.is_file():\n        try:\n            meta = json.loads(pointer.read_text(encoding="utf-8"))\n            key = str(meta.get("cache_key") or "").strip()\n            if key:\n                b = cache_dir / key\n                s = cache_dir / f"{key}.sha256"\n                if verify_cached_binary(b, s):\n                    return b, s, key\n        except (OSError, ValueError):\n            pass\n    return None\n\n\ndef _ensure_rust_toolchain(log: Callable[[str], None]) -> dict[str, str]:\n    cargo_bin = Path.home() / ".cargo" / "bin"\n    env = dict(os.environ)\n    env["PATH"] = f"{cargo_bin}:{env.get(\'PATH\', \'\')}"\n    if shutil.which("cargo", path=env["PATH"]) and shutil.which("rustc", path=env["PATH"]):\n        ver = subprocess.check_output(["rustc", "--version"], env=env, text=True, timeout=15).strip()\n        log(f"Using existing Rust toolchain: {ver}")\n        return env\n\n    log("Installing minimal Rust toolchain via rustup...")\n    t0 = time.monotonic()\n    cmd = "curl --proto \'=https\' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal --default-toolchain stable"\n    subprocess.run(["bash", "-c", cmd], env=env, check=True, timeout=300)\n    ver = subprocess.check_output(["rustc", "--version"], env=env, text=True, timeout=15).strip()\n    log(f"Installed Rust toolchain in {time.monotonic() - t0:.1f}s: {ver}")\n    return env\n\n\ndef _run_streaming(cmd: list[str], *, env: dict[str, str] | None = None, timeout: int = 1500, log: Callable[[str], None] = print) -> None:\n    """Run a command and stream its combined output through `log` (visible in Jupyter/Colab)."""\n    proc = subprocess.Popen(\n        cmd,\n        env=env,\n        stdout=subprocess.PIPE,\n        stderr=subprocess.STDOUT,\n        text=True,\n        bufsize=1,\n    )\n    deadline = time.monotonic() + timeout\n    assert proc.stdout is not None\n    try:\n        for line in proc.stdout:\n            text = line.rstrip()\n            if text:\n                log(text)\n            if time.monotonic() > deadline:\n                proc.kill()\n                raise subprocess.TimeoutExpired(cmd, timeout)\n        rc = proc.wait(timeout=max(1, int(deadline - time.monotonic())))\n    except BaseException:\n        proc.kill()\n        proc.wait()\n        raise\n    if rc != 0:\n        raise subprocess.CalledProcessError(rc, cmd)\n\n\ndef _build_binary(\n    spec: dict[str, str],\n    build_root: Path,\n    *,\n    lto: str = "thin",\n    log: Callable[[str], None] = print,\n) -> Path:\n    env = _ensure_rust_toolchain(log)\n    lto_norm = (lto or "thin").strip().lower()\n    if lto_norm in ("thin", "off", "false"):\n        env["CARGO_PROFILE_RELEASE_LTO"] = lto_norm\n        env["CARGO_PROFILE_RELEASE_CODEGEN_UNITS"] = "16" if lto_norm in ("off", "false") else "4"\n\n    shutil.rmtree(build_root, ignore_errors=True)\n    build_root.mkdir(parents=True, exist_ok=True)\n    t0 = time.monotonic()\n\n    if spec["kind"] == "crates":\n        install_root = build_root / "cargo-root"\n        cmd = ["cargo", "install", "firecrawl-local", "--locked", "--root", str(install_root)]\n        if spec["ref"] and spec["ref"] != "latest":\n            cmd.extend(["--version", spec["ref"]])\n        log(f"Building firecrawl-local from crates.io (version={spec[\'ref\']}, lto={lto_norm})...")\n        _run_streaming(cmd, env=env, timeout=1500, log=log)\n        built = install_root / "bin" / "firecrawl-local"\n    else:\n        repo_dir = build_root / "firecrawl"\n        checkout_ref = spec.get("checkout_ref") or spec["ref"] or "main"\n        log(f"Sparse-cloning firecrawl/firecrawl ({checkout_ref}) for apps/local + apps/api/native/src...")\n        if _HEX_SHA_RE.match(checkout_ref):\n            subprocess.run(["git", "init", str(repo_dir)], check=True, timeout=30)\n            subprocess.run(["git", "-C", str(repo_dir), "remote", "add", "origin", FIRECRAWL_REPO_URL], check=True, timeout=30)\n            subprocess.run(["git", "-C", str(repo_dir), "sparse-checkout", "init", "--cone"], check=True, timeout=30)\n            subprocess.run(["git", "-C", str(repo_dir), "sparse-checkout", "set", "apps/local", "apps/api/native/src"], check=True, timeout=30)\n            subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth=1", "--filter=blob:none", "origin", checkout_ref], check=True, timeout=180)\n            subprocess.run(["git", "-C", str(repo_dir), "checkout", "FETCH_HEAD"], check=True, timeout=60)\n        else:\n            subprocess.run(\n                [\n                    "git", "clone", "--filter=blob:none", "--no-checkout", "--depth=1",\n                    "--branch", checkout_ref, FIRECRAWL_REPO_URL, str(repo_dir),\n                ],\n                check=True,\n                timeout=180,\n            )\n            subprocess.run(["git", "-C", str(repo_dir), "sparse-checkout", "init", "--cone"], check=True, timeout=30)\n            subprocess.run(["git", "-C", str(repo_dir), "sparse-checkout", "set", "apps/local", "apps/api/native/src"], check=True, timeout=30)\n            subprocess.run(["git", "-C", str(repo_dir), "checkout"], check=True, timeout=60)\n\n        target_dir = build_root / "target"\n        env["CARGO_TARGET_DIR"] = str(target_dir)\n        log(f"Compiling firecrawl-local release binary (lto={lto_norm}, cpus={os.cpu_count()})...")\n        _run_streaming(\n            ["cargo", "build", "--release", "--locked", "--manifest-path", str(repo_dir / "apps/local/Cargo.toml")],\n            env=env,\n            timeout=1500,\n            log=log,\n        )\n        built = target_dir / "release" / "firecrawl-local"\n\n    if not built.is_file():\n        raise RuntimeError(f"Expected compiled binary at {built}, but file was not produced.")\n    log(f"Compiled firecrawl-local in {time.monotonic() - t0:.1f}s ({built.stat().st_size / 1e6:.1f} MB)")\n    return built\n\n\ndef _probe_binary_version(binary_path: Path) -> str:\n    return subprocess.check_output([str(binary_path), "--version"], text=True, timeout=15).strip()\n\n\ndef ensure_binary(\n    work_dir: Path | str,\n    data_dir: Path | str,\n    *,\n    ref: str = "main",\n    rebuild: bool = False,\n    lto: str = "thin",\n    build_root: Path | str = "/content/build/firecrawl-local",\n    log: Callable[[str], None] = print,\n) -> dict[str, str]:\n    """Ensure `/content/firecrawl-local/bin/firecrawl-local` is ready.\n\n    Checks local SSD -> Google Drive cache -> builds on local SSD and caches to Drive.\n    """\n    work_path = Path(work_dir)\n    data_path = Path(data_dir)\n    local_bin = work_path / "bin" / "firecrawl-local"\n    local_stamp = work_path / "bin" / ".cache_key"\n    cache_dir = data_path / "bin"\n    arch = detect_arch()\n    glibc = detect_glibc_version()\n\n    spec = resolve_source_spec(ref, log=log)\n    cache_key = make_cache_key(spec["cache_tag"], arch=arch, glibc=glibc)\n    cached_bin = cache_dir / cache_key\n    cached_sha = cache_dir / f"{cache_key}.sha256"\n\n    if not rebuild and local_bin.is_file() and local_stamp.is_file():\n        if local_stamp.read_text(encoding="utf-8").strip() == cache_key:\n            try:\n                ver = _probe_binary_version(local_bin)\n                log(f"Local binary already up to date ({cache_key}): {ver}")\n                return {"binary": str(local_bin), "cache_key": cache_key, "version": ver, "source": "local"}\n            except (OSError, subprocess.SubprocessError):\n                pass\n\n    if not rebuild and verify_cached_binary(cached_bin, cached_sha):\n        t0 = time.monotonic()\n        _atomic_copy(cached_bin, local_bin, mode=0o755)\n        try:\n            ver = _probe_binary_version(local_bin)\n            _atomic_write_text(local_stamp, cache_key + "\\n")\n            log(f"Restored binary from Drive cache in {time.monotonic() - t0:.2f}s ({cache_key}): {ver}")\n            return {"binary": str(local_bin), "cache_key": cache_key, "version": ver, "source": "drive-cache"}\n        except (OSError, subprocess.SubprocessError) as err:\n            log(f"Warning: Drive cached binary failed verification ({err}); rebuilding.")\n            local_bin.unlink(missing_ok=True)\n\n    # If remote lookup fell back to unhashed branch name, reuse latest valid Drive cache when not rebuilding.\n    if not rebuild and spec["cache_tag"] == f"git-{sanitize_token(ref, \'main\')}":\n        fb = _find_latest_fallback_cache(cache_dir, arch, glibc)\n        if fb is not None:\n            fb_bin, _, fb_key = fb\n            _atomic_copy(fb_bin, local_bin, mode=0o755)\n            ver = _probe_binary_version(local_bin)\n            _atomic_write_text(local_stamp, fb_key + "\\n")\n            log(f"Restored fallback binary from Drive cache ({fb_key}): {ver}")\n            return {"binary": str(local_bin), "cache_key": fb_key, "version": ver, "source": "drive-cache-fallback"}\n\n    built = _build_binary(spec, Path(build_root), lto=lto, log=log)\n    _atomic_copy(built, local_bin, mode=0o755)\n    ver = _probe_binary_version(local_bin)\n    _atomic_write_text(local_stamp, cache_key + "\\n")\n\n    # Persist to Drive cache atomically.\n    try:\n        digest = sha256_file(local_bin)\n        _atomic_copy(local_bin, cached_bin)\n        _atomic_write_text(cached_sha, f"{digest}  {cache_key}\\n")\n        _atomic_write_text(\n            cache_dir / f"latest-{arch}-glibc{glibc}.json",\n            json.dumps({"cache_key": cache_key, "version": ver, "ref": ref, "sha256": digest}, indent=2) + "\\n",\n        )\n        log(f"Saved compiled binary to Drive cache: {cached_bin}")\n    except OSError as err:\n        log(f"Warning: failed to persist binary to Drive cache ({err}); continuing with local SSD copy.")\n    finally:\n        shutil.rmtree(build_root, ignore_errors=True)\n\n    return {"binary": str(local_bin), "cache_key": cache_key, "version": ver, "source": "built"}\n\n\ndef ensure_nonroot_user(username: str = "firecrawl", work_dir: Path | str = "/content/firecrawl-local") -> dict[str, str]:\n    """Ensure an unprivileged user exists for running firecrawl-local + sandboxed Chrome."""\n    work_path = Path(work_dir)\n    if subprocess.run(["id", "-u", username], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:\n        subprocess.run(["useradd", "-m", "-s", "/bin/bash", username], check=True, timeout=30)\n\n    home_dir = work_path / "home"\n    tmp_dir = work_path / "tmp"\n    work_path.mkdir(parents=True, exist_ok=True)\n    os.chmod(work_path, 0o755)\n    if (work_path / "bin").exists():\n        os.chmod(work_path / "bin", 0o755)\n    for d in (home_dir, tmp_dir):\n        d.mkdir(parents=True, exist_ok=True)\n        subprocess.run(["chown", "-R", f"{username}:{username}", str(d)], check=True, timeout=30)\n        os.chmod(d, 0o700)\n\n    return {"user": username, "home": str(home_dir), "tmp": str(tmp_dir)}\n\n\ndef ensure_chrome(browser_mode: str = "auto", *, log: Callable[[str], None] = print) -> str | None:\n    """Install google-chrome-stable when browser_mode != \'never\'."""\n    mode = (browser_mode or "auto").strip().lower()\n    if mode == "never":\n        log("Browser mode is \'never\'; skipping Chrome installation.")\n        return None\n\n    for candidate in ("/usr/bin/google-chrome-stable", "/usr/bin/google-chrome", "/usr/bin/chromium", "/usr/bin/chromium-browser"):\n        if Path(candidate).is_file():\n            ver = subprocess.check_output([candidate, "--version"], text=True, timeout=15).strip()\n            log(f"Using existing browser: {ver} ({candidate})")\n            return candidate\n\n    log("Installing google-chrome-stable for native-sandbox headless JS rendering...")\n    t0 = time.monotonic()\n    deb_path = Path("/tmp/google-chrome-stable.deb")\n    try:\n        subprocess.run(["curl", "-fsSL", "-o", str(deb_path), CHROME_DEB_URL], check=True, timeout=180)\n        subprocess.run(["apt-get", "update", "-qq"], check=True, timeout=120, stdout=subprocess.DEVNULL)\n        env = dict(os.environ, DEBIAN_FRONTEND="noninteractive")\n        subprocess.run(\n            ["apt-get", "install", "-y", "-qq", str(deb_path)],\n            env=env,\n            check=True,\n            timeout=300,\n            stdout=subprocess.DEVNULL,\n        )\n    finally:\n        deb_path.unlink(missing_ok=True)\n\n    chrome = "/usr/bin/google-chrome-stable"\n    if not Path(chrome).is_file():\n        raise RuntimeError("google-chrome-stable installation completed, but /usr/bin/google-chrome-stable is missing.")\n    ver = subprocess.check_output([chrome, "--version"], text=True, timeout=15).strip()\n    log(f"Installed {ver} in {time.monotonic() - t0:.1f}s")\n    return chrome\n', encoding="utf-8")

if str(FC_WORK) not in sys.path:
    sys.path.insert(0, str(FC_WORK))
import firecrawl_local_setup as _fc_setup

def _fc_setup_log(msg):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] [firecrawl-setup] {msg}", flush=True)

FC_REF = os.environ.get('CT_FIRECRAWL_REF', 'main').strip() or 'main'
FC_REBUILD = os.environ.get('CT_FIRECRAWL_REBUILD', '0').strip() in ('1', 'true', 'yes')
FC_LTO = os.environ.get('CT_FIRECRAWL_LTO', 'thin').strip() or 'thin'
FC_BROWSER = os.environ.get('CT_FIRECRAWL_BROWSER', 'auto').strip().lower() or 'auto'
FC_CLOUD = os.environ.get('CT_FIRECRAWL_CLOUD', 'off').strip().lower() or 'off'

FC_BIN_INFO = _fc_setup.ensure_binary(
    FC_WORK, Path(DATA_DIR), ref=FC_REF, rebuild=FC_REBUILD, lto=FC_LTO, log=_fc_setup_log
)
FC_USER_INFO = _fc_setup.ensure_nonroot_user('firecrawl', FC_WORK)
FC_CHROME_PATH = _fc_setup.ensure_chrome(FC_BROWSER, log=_fc_setup_log)
FC_BIN = FC_BIN_INFO['binary']
print(
    f"firecrawl-local ready: {FC_BIN_INFO['version']} (source={FC_BIN_INFO['source']}, "
    f"cache_key={FC_BIN_INFO['cache_key']}, browser={FC_BROWSER}, chrome={FC_CHROME_PATH or 'none'}, "
    f"cloud={FC_CLOUD}, cpus={os.cpu_count()}, free_ssd={shutil.disk_usage('/content').free / 1e9:.1f}GB)",
    flush=True,
)


In [ ]:
# @title 2. Service -- start loopback Firecrawl Local v2 API as unprivileged `firecrawl` user
if "FC_BIN" not in globals():
    raise RuntimeError("firecrawl-local setup cell did not run; run Cell 1 first.")

subprocess.run(["pkill", "-f", str(FC_BIN)], check=False)

def start_service():
    fc_env = {
        "HOME": FC_USER_INFO["home"],
        "TMPDIR": FC_USER_INFO["tmp"],
        "FIRECRAWL_LOCAL_HOME": FC_USER_INFO["home"],
        "FIRECRAWL_LOCAL_BROWSER": FC_BROWSER,
        "FIRECRAWL_CLOUD": FC_CLOUD,
    }
    if FC_CHROME_PATH:
        fc_env["FIRECRAWL_CHROME_PATH"] = FC_CHROME_PATH
    ua = os.environ.get("CT_FIRECRAWL_USER_AGENT", "").strip()
    if ua:
        fc_env["FIRECRAWL_LOCAL_USER_AGENT"] = ua
    if os.environ.get("CT_FIRECRAWL_SCROLL", "0").strip() in ("1", "true", "yes"):
        fc_env["FIRECRAWL_LOCAL_SCROLL"] = "1"
    fc_key = os.environ.get("FIRECRAWL_API_KEY", "").strip()
    if fc_key:
        fc_env["FIRECRAWL_API_KEY"] = fc_key
    whitelist = ",".join(sorted(fc_env.keys()))
    cmd = [
        "runuser",
        "-w", whitelist,
        "-u", FC_USER_INFO["user"],
        "--",
        str(FC_BIN),
        "serve",
        "--host", "127.0.0.1",
        "--port", str(SERVICE_PORT),
    ]
    return ct_spawn("service", cmd, cwd=FC_USER_INFO["home"], env=fc_env)

PROCS["service"] = (start_service(), start_service)
ct_wait_http(f"http://127.0.0.1:{SERVICE_PORT}/health", timeout=60, ok=lambda c: c == 200)
print(f"firecrawl-local API listening on 127.0.0.1:{SERVICE_PORT} (user={FC_USER_INFO['user']})", flush=True)


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = True   # True: send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)

def start_caddy():
    caddy_bin = ct_ensure_caddy()
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output([caddy_bin, "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            header_up -Origin
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                "{AUTH_USER}" {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    ct_write_private_text(f"{WORK}/Caddyfile", caddyfile)
    return ct_spawn_marked("caddy", [caddy_bin, "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

CLOUDFLARED_VERSION = '2026.11.0'
CLOUDFLARED_SHA256 = {
    'amd64': '51f7c68b57ed7269164075d7e789efce5b59355982f06392deeb037d982a51fd',
    'arm64': 'abe54a006acf9faf2d19ee9d953cb0d0804e92f59c5382a162e57b84d086f56f',
}


def _write_private_text(path, content):
    path = Path(path)
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    os.chmod(path, 0o600)
    with os.fdopen(fd, 'w', encoding='utf-8') as handle:
        handle.write(content)
        handle.flush()
        os.fsync(handle.fileno())


def _touch_private_log(path):
    path = Path(path)
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    os.chmod(path, 0o600)
    os.close(fd)


def _file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        while True:
            chunk = handle.read(65536)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def _check_cached_binary(destination, artifact, version, cache_dir):
    if not destination.exists():
        return None
    if not cache_dir:
        return 'hit'
    manifest = Path(cache_dir) / 'manifests' / f'{artifact}.json'
    if not manifest.is_file():
        return 'miss'
    try:
        meta = json.loads(manifest.read_text(encoding='utf-8'))
        if (
            meta.get('artifact') == artifact
            and meta.get('version') == version
            and meta.get('sha256') == _file_sha256(destination)
        ):
            return 'hit'
    except (OSError, ValueError):
        pass
    destination.unlink(missing_ok=True)
    return 'corrupt_rebuilt'


def _write_manifest(cache_dir, artifact, version, destination):
    if not cache_dir:
        return
    manifest_dir = Path(cache_dir) / 'manifests'
    manifest_dir.mkdir(parents=True, exist_ok=True)
    manifest = manifest_dir / f'{artifact}.json'
    tmp = manifest.with_name(f'.{artifact}.{os.getpid()}.tmp')
    payload = {
        'artifact': artifact,
        'version': version,
        'sha256': _file_sha256(destination),
        'bin_path': str(destination),
        'verified_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }
    tmp.write_text(json.dumps(payload, indent=2) + '\n', encoding='utf-8')
    os.replace(tmp, manifest)


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version, *, cache_dir=None):
    destination = Path(destination)
    cache_state = _check_cached_binary(destination, 'frpc', version, cache_dir)
    if cache_state == 'hit':
        if cache_dir:
            print(f'CT_CACHE_EVENT artifact=frpc status=hit', flush=True)
        return str(destination)
    status_label = cache_state or 'miss'
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            partial=destination.with_name(f'{destination.name}.{os.getpid()}.partial')
            try:
                with tar.extractfile(entry) as source, open(partial, 'wb') as out_bin:
                    shutil.copyfileobj(source, out_bin)
                partial.chmod(0o755)
                os.replace(partial, destination)
            finally:
                partial.unlink(missing_ok=True)
    _write_manifest(cache_dir, 'frpc', version, destination)
    if cache_dir:
        print(f'CT_CACHE_EVENT artifact=frpc status={status_label}', flush=True)
    return str(destination)


def install_cloudflared(destination, *, cache_dir=None, fetch_fn=None):
    destination = Path(destination)
    cf_arch = {'x86_64': 'amd64', 'aarch64': 'arm64'}.get(platform.machine(), 'amd64')
    url = f'https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-{cf_arch}'
    expected_sha = CLOUDFLARED_SHA256.get(cf_arch)
    if not expected_sha:
        raise ValueError(f'Unsupported cloudflared architecture: {platform.machine()}')
    if cache_dir:
        cache_state = _check_cached_binary(destination, 'cloudflared', CLOUDFLARED_VERSION, cache_dir)
        if cache_state == 'hit':
            print('CT_CACHE_EVENT artifact=cloudflared status=hit', flush=True)
            return str(destination)
        status_label = cache_state or 'miss'
        if destination.exists() and _file_sha256(destination) != expected_sha:
            destination.unlink(missing_ok=True)
            status_label = 'corrupt_rebuilt'
        destination.parent.mkdir(parents=True, exist_ok=True)
        partial = destination.with_name(f'{destination.name}.{os.getpid()}.partial')
        try:
            with urllib.request.urlopen(url, timeout=60) as src, open(partial, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            if _file_sha256(partial) != expected_sha:
                raise ValueError('cloudflared checksum verification failed')
            partial.chmod(0o755)
            os.replace(partial, destination)
        finally:
            partial.unlink(missing_ok=True)
        _write_manifest(cache_dir, 'cloudflared', CLOUDFLARED_VERSION, destination)
        print(f'CT_CACHE_EVENT artifact=cloudflared status={status_label}', flush=True)
        return str(destination)
    if fetch_fn is not None:
        return fetch_fn(str(destination), url)
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        partial = destination.with_name(f'{destination.name}.{os.getpid()}.partial')
        try:
            with urllib.request.urlopen(url, timeout=60) as src, open(partial, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            if _file_sha256(partial) != expected_sha:
                raise ValueError('cloudflared checksum verification failed')
            partial.chmod(0o755)
            os.replace(partial, destination)
        finally:
            partial.unlink(missing_ok=True)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx.get('WORK') or os.environ.get('CT_WORK_DIR') or '/content/ct')
    work.mkdir(parents=True, exist_ok=True)
    bin_dir_explicit = ctx.get('BIN_DIR') or os.environ.get('CT_BIN_DIR') or ''
    bin_dir = Path(bin_dir_explicit or '/usr/local/bin')
    cache_dir = ctx.get('CACHE_DIR') or os.environ.get('CT_CACHE_DIR') or ''
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if ctx.get('ct_stop_marked_proc') is not None:
        stop_fn = ctx['ct_stop_marked_proc']
        stop_fn('tunnel-process')
    elif marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        _touch_private_log(work/(name+'.log'))
        proc=ctx['ct_spawn'](name,args)
        _write_private_text(marker, json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        _write_private_text(ca_path, os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        _write_private_text(config_path, render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        frpc_name = f"frpc-{os.environ['CT_FRP_VERSION']}"
        frpc_dest = (bin_dir / frpc_name) if bin_dir_explicit else ((work / frpc_name) if (work / frpc_name).exists() else (bin_dir / frpc_name))
        binary=install_frpc(frpc_dest, os.environ['CT_FRP_VERSION'], cache_dir=cache_dir or None)
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        binary=install_cloudflared(bin_dir / 'cloudflared', cache_dir=cache_dir or None, fetch_fn=ctx.get('ct_fetch_binary'))
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see {work}/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Ready + keep-alive -- verify public Caddy Basic/Bearer auth before CT_READY
def _fc_log(msg):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] [firecrawl-local] {msg}", flush=True)

def _fc_public_probe():
    req = urllib.request.Request(
        PUBLIC_URL + "/health",
        headers={"Authorization": f"Bearer {API_TOKEN}", "User-Agent": "Mozilla/5.0"},
    )
    with urllib.request.urlopen(req, timeout=20) as resp:
        data = json.loads(resp.read().decode("utf-8"))
        if data.get("status") != "ok":
            raise RuntimeError(f"Public /health returned unexpected payload: {data}")
        return resp.status

def _fc_verify_auth():
    req_unauth = urllib.request.Request(PUBLIC_URL + "/health", headers={"User-Agent": "Mozilla/5.0"})
    try:
        urllib.request.urlopen(req_unauth, timeout=15)
        raise RuntimeError("Unauthenticated public GET /health unexpectedly succeeded!")
    except urllib.error.HTTPError as err:
        if err.code != 401:
            raise RuntimeError(f"Expected HTTP 401 without credentials, got {err.code}")
        if err.headers.get("Set-Cookie"):
            raise RuntimeError("Public 401 leaked Set-Cookie")
    req_bad = urllib.request.Request(
        PUBLIC_URL + "/health",
        headers={"Authorization": "Bearer invalid-token", "User-Agent": "Mozilla/5.0"},
    )
    try:
        urllib.request.urlopen(req_bad, timeout=15)
        raise RuntimeError("Invalid Bearer token unexpectedly succeeded!")
    except urllib.error.HTTPError as err:
        if err.code != 401:
            raise RuntimeError(f"Expected HTTP 401 for bad Bearer token, got {err.code}")

ct_public_gate(PUBLIC_URL, _fc_public_probe, _fc_verify_auth, _fc_log, timeout=180, interval=8)
if not CREDS_FROM_ENV:
    print(f"CT_CREDENTIALS user={AUTH_USER} pass={AUTH_PASS}")
    print(f"API token: {API_TOKEN}")
print(f"{SERVICE} is up: {PUBLIC_URL}")
print("CT_READY", flush=True)

def _fc_local_health():
    with urllib.request.urlopen(f"http://127.0.0.1:{SERVICE_PORT}/health", timeout=10) as resp:
        return resp.status == 200 and json.loads(resp.read().decode("utf-8")).get("status") == "ok"

def _fc_heartbeat(uptime_s):
    with urllib.request.urlopen(f"http://127.0.0.1:{SERVICE_PORT}/health", timeout=10) as resp:
        info = json.loads(resp.read().decode("utf-8"))
    _fc_log(
        f"heartbeat | up={uptime_s}s | version={info.get('version', '?')} | "
        f"browser={FC_BROWSER} | cloud={FC_CLOUD} | "
        f"free_ssd={shutil.disk_usage('/content').free / 1e9:.1f}GB"
    )

ct_watchdog(
    _fc_local_health,
    start_service,
    _fc_log,
    service_key="service",
    log_name="service",
    grace=60,
    heartbeat=_fc_heartbeat,
)
